# 1. Data collection

 - Employment Ordinance
 - Minimum Wage
 - OSH Ordinance
 - FDH Rights 
 - Practical Guide for Operating an Employment Agency (Relevant Chapters)
     - Introduction to Part XII of Emplyment Ordinance
     - Do's and Dont's 
     - FAQ
 - Code of Practice for Employment Agencies
 - FDH Corner (html webpage) scraped below
 - Hiring FDH Guidebook (htm webpage) scraped below - MAW, food allowance

In [2]:
import os
from langchain_community.document_loaders import WebBaseLoader
import requests
from bs4 import BeautifulSoup

# Set Chrome User-Agent
os.environ['USER_AGENT'] = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

print("\nScraping FDW Corner webpage for content")
url = "https://www.fdh.labour.gov.hk/en/fdh_corner.html"
response = requests.get(url, timeout=30)
soup = BeautifulSoup(response.content, 'html.parser')

# Get all text and clean it
text = soup.get_text()
lines = [line.strip() for line in text.split('\n') if line.strip()]

# Filter out navigation and short lines
filtered = [line for line in lines if len(line) > 20 and not any(x in line.lower() for x in ['skip', 'menu', 'search'])]

with open("../data/raw/fdw_corner_webpage.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(filtered))

print(f"✓ Saved {len(filtered)} lines to data/raw/fdw_corner_webpage.txt")
print(f"\nPreview:\n{filtered[0][:200]}...")


Scraping FDW Corner webpage for content
✓ Saved 37 lines to data/raw/fdw_corner_webpage.txt

Preview:
Foreign Domestic Helpers - Foreign Domestic Helpers' Corner...


In [21]:
import os
from langchain_community.document_loaders import WebBaseLoader
import requests
from bs4 import BeautifulSoup

# Set Chrome User-Agent
os.environ['USER_AGENT'] = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

print("\nScraping guidebook to hire FDH for content")
url = "https://www.gov.hk/en/theme/guidebook/employment/family/foreigndomestichelper.htm"
response = requests.get(url, timeout=30)
soup = BeautifulSoup(response.content, 'html.parser')

# Get all text and clean it
text = soup.get_text()
lines = [line.strip() for line in text.split('\n') if line.strip()]

# Filter out navigation and short lines
filtered = [line for line in lines if len(line) > 20 and not any(x in line.lower() for x in ['skip', 'menu', 'search'])]

with open("../data/raw/fdh_hire_guidebook.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(filtered))

print(f"✓ Saved {len(filtered)} lines to data/raw/fdh_hire_guidebook.txt")
print(f"\nPreview:\n{filtered[0][:200]}...")


Scraping guidebook to hire FDH for content
✓ Saved 185 lines to data/raw/fdh_hire_guidebook.txt

Preview:
GovHK: Hiring Foreign Domestic Helpers (FDHs)...


In [25]:
import re

# Read the file
with open('../data/raw/fdh_hire_guidebook.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Remove unnecessary navigation content
pattern = r'(Who is eligible to hire an FDH?.*?)(?=Found this page helpful|\Z)'
match = re.search(pattern, text, re.DOTALL)
if match:
    clean = match.group(1)

with open("../data/raw/fdh_hire_guidebook.txt", "w", encoding="utf-8") as f:
    f.write(clean)

In [1]:
# 8 PDFs manually downloaded from source

# 2. Data Preprocessing (store chunks)

In [ ]:
# preprocess.py - RUN FILE FROM CLI, NOTEBOOK ONLY FOR DISPLAY (libraries in venv)
import os
import re
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

class LaborDocumentsPreprocessor:
    def __init__(self, pdf_dir="../data/raw", txt_file="../data/raw/fdw_corner_webpage.txt", output_dir="../data/processed"):
        self.pdf_dir = pdf_dir
        self.txt_file = txt_file
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def clean_text(self, text):
        """
        Removes irrelevant content and cleans text
        """
        # Removing extra whitespace and newlines
        text = re.sub(r'\n\s*\n', '\n\n', text)  # Normalize paragraph breaks
        text = re.sub(r' +', ' ', text)           # Remove multiple spaces
        text = text.strip()
        
        # Removing page numbers
        text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
        text = re.sub(r'Page \d+ of \d+', '', text)
        
        # Saving clean lines
        lines = text.split('\n')
        cleaned_lines = []
        for line in lines:
            line = line.strip()
            # Skipping navigation
            if len(line) < 15 and ('Labour Department' in line or 'www' in line):
                continue
            if line.isdigit():
                continue
            cleaned_lines.append(line)
        
        return '\n'.join(cleaned_lines)
    
    def load_and_clean_pdfs(self):
        """Load all PDFs -> extract and clean"""
        all_docs = []
        
        pdf_files = [f for f in os.listdir(self.pdf_dir) if f.endswith('.pdf')]
        print(f"Found {len(pdf_files)} PDF files")
        
        for pdf_file in pdf_files:
            print(f"Processing: {pdf_file}")
            try:
                loader = PyPDFLoader(os.path.join(self.pdf_dir, pdf_file))
                pages = loader.load()
                full_text = "\n".join([page.page_content for page in pages])
                cleaned_text = self.clean_text(full_text)
                all_docs.append(Document(
                    page_content=cleaned_text,
                    metadata={"source": pdf_file, "type": "pdf"}
                ))
                print(f"Extracted {len(cleaned_text)} chars")
            except Exception as e:
                print(f"Error: {e}")
        
        return all_docs
    
    def load_and_clean_txt(self):
        """Loads and cleans the scraped HTML text file"""
        print(f"\nProcessing txt file: {self.txt_file}")
        
        try:
            with open(self.txt_file, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # Clean extra spaces and normalize
            cleaned = re.sub(r'\n\s*\n', '\n\n', text)
            cleaned = re.sub(r' +', ' ', cleaned)
            cleaned = cleaned.strip()
            
            doc = Document(
                page_content=cleaned,
                metadata={"source": "fdw_corner_webpage.txt", "type": "html"}
            )
            print(f"Loaded {len(cleaned)} chars")
            return [doc]
        except Exception as e:
            print(f"Error: {e}")
            return []
    
    def chunk_documents(self, documents, chunk_size=500, chunk_overlap=50):
        """Splits documents into smaller chunks for retrieval"""
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", ", ", " ", ""],
            length_function=len,
        )
        
        chunks = text_splitter.split_documents(documents)
        print(f"\nSplit into {len(chunks)} chunks with size={chunk_size} and overlap={chunk_overlap}")
        
        # Save chunks to file for inspection
        output_file = os.path.join(self.output_dir, "all_chunks.txt")
        with open(output_file, 'w', encoding='utf-8') as f:
            for i, chunk in enumerate(chunks):
                f.write(f"\n{'='*60}\n")
                f.write(f"CHUNK {i+1} (Source: {chunk.metadata.get('source', 'unknown')})\n")
                f.write(f"{'='*60}\n")
                f.write(chunk.page_content)
                f.write("\n")
        
        print(f"Chunks saved to: {output_file}")
        return chunks
    
    def save_chunks_for_rag(self, chunks):
        """Saves chunks as JSON for later use in RAG pipeline"""
        import json
        
        chunk_data = []
        for chunk in chunks:
            chunk_data.append({
                "content": chunk.page_content,
                "source": chunk.metadata.get("source", "unknown"),
                "type": chunk.metadata.get("type", "unknown")
            })
        
        json_path = os.path.join(self.output_dir, "processed_chunks.json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(chunk_data, f, ensure_ascii=False, indent=2)
        
        print(f"JSON saved to: {json_path}")
        return chunk_data
    
    def run(self):
        """Full preprocessing pipeline"""
        print("="*60)
        print("LABOR DOCUMENT PREPROCESSING")
        print("="*60)

        # Loading and cleaning all docs
        pdf_docs = self.load_and_clean_pdfs()
        txt_docs = self.load_and_clean_txt()
        
        # Combining all docs
        all_docs = pdf_docs + txt_docs
        print(f"\nTotal documents loaded: {len(all_docs)}")
        
        # Creating chunks
        chunks = self.chunk_documents(all_docs)
        
        # 5. Saving for RAG
        self.save_chunks_for_rag(chunks)
        
        print("\n" + "="*60)
        print("PREPROCESSING COMPLETE!")
        print(f"Output saved to: {self.output_dir}")
        print("="*60)
        
        return chunks

if __name__ == "__main__":
    preprocessor = LaborDocumentsPreprocessor()
    chunks = preprocessor.run()
    # Preview
    print("\nCHUNK PREVIEW (first 3 chunks):")
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(chunk.page_content[:300] + "...")

In [ ]:
""" From CLI:
============================================================
LABOR DOCUMENT PREPROCESSING
============================================================
Found 8 PDF files
Processing: Concise Guide Employment Ordinance.pdf
Extracted 85752 chars
Processing: Concise Guide Minimum Wage.pdf
Extracted 17950 chars
Processing: CoP_EA_Eng.pdf
Extracted 92705 chars
Processing: FDHguideEnglish.pdf
Extracted 57485 chars
Processing: OSH HK.pdf
Extracted 38539 chars
Processing: PGEA_Chapter_1.pdf
Extracted 10897 chars
Processing: PGEA_Chapter_3.pdf
Extracted 3387 chars
Processing: PGEA_Chapter_4.pdf
Extracted 7790 chars

Processing txt file: ../data/raw/fdw_corner_webpage.txt
Loaded 4866 chars

Total documents loaded: 9

Split into 787 chunks with size=500 and overlap=50
Chunks saved to: ../data/processed\all_chunks.txt
JSON saved to: ../data/processed\processed_chunks.json

============================================================
PREPROCESSING COMPLETE!
Output saved to: ../data/processed
============================================================

CHUNK PREVIEW (first 3 chunks):

--- Chunk 1 ---
A Concise Guide to the
Employment Ordinance

Labour Department
Table of Content

Foreword
Chapter 1

Application of the Employment Ordinance 4
Chapter 2

Contract of Employment 5
Chapter 3

Wages 8
Chapter 4

R est Days, Holidays and Leave 12
Chapter 5

Sickness Allowance 20
Chapter 6

Maternity Pro...

--- Chunk 2 ---
Employment Protection 36
Chapter 10

Severance Payment and Long Service Payment 40
Chapter 11

Protection against Anti -union Discrimination 46
Chapter 12

Employers’ Criminal Liability in Failing to Pay an Award
of the Labour Tribunal or Minor Employment Claims
Adjudication Board
Appendix 1

A Guid...

--- Chunk 3 ---
Enquiries
Foreword
This guide sets out in simple terms the main provisions of the Employment
Ordinance (Cap. 57). It should be noted that the Ordinance itself remains the
sole authority for the provisions of the law explained. Please refer to the
Appendix 2 for enquiries services.
Note:
The Em
ploym...
"""

# 3. Embedding and Retrieval Setup

In [ ]:
# embedding_setup.py - RUN FILE FROM CLI, NOTEBOOK ONLY FOR DISPLAY (libraries in venv)
import os
import json
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

# Loading chunks
def load_chunks(json_path="../data/processed/processed_chunks.json"):
    """Loads chunks saved from preprocessing step"""
    
    with open(json_path, 'r', encoding='utf-8') as f:
        chunk_data = json.load(f)
    
    # Converting back to Document objects
    documents = []
    for item in chunk_data:
        doc = Document(
            page_content=item["content"],
            metadata={
                "source": item["source"],
                "type": item.get("type", "unknown")
            }
        )
        documents.append(doc)

    print(f"Loaded {len(documents)} chunks from {json_path}")
    return documents

# Embeddings and vectorstore
print("Creating embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Load chunks
chunks = load_chunks()

# Build FAISS index
print("Building FAISS index")
vectorstore = FAISS.from_documents(chunks, embeddings)

# Saving
os.makedirs("../vectorstore", exist_ok=True)
vectorstore.save_local("../vectorstore/faiss_index")
print(f"Saved vectorstore with {len(chunks)} chunks")

# Creating retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Testing retrieval
def test_retrieval(query):
    results = retriever.invoke(query)
    print(f"Query: {query}")
    print(f"Retrieved {len(results)} chunks:\n")
    for i, doc in enumerate(results, 1):
        print(f"{i}. Source: {doc.metadata['source']}")
        print(f"Content: {doc.page_content[:150]}...\n")
    return results

if __name__ == "__main__":
    # Some test queries
    test_retrieval("What are the rights of domestic workers?")
    test_retrieval("What is the minimum wage?")
    test_retrieval("What are the rules for employment agencies?")

In [ ]:
""" From CLI:
Creating embeddings...
C:\Work\Migrasia\PoBot\notebooks\embedding_setup.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loaded 787 chunks from ../data/processed/processed_chunks.json
Building FAISS index
Saved vectorstore with 787 chunks
Query: What are the rights of domestic workers?
Retrieved 3 chunks:

1. Source: FDHguideEnglish.pdf
Content: Ordinance which remains the sole authority for the provisions
explained. The court is the authority on questions of interpretation
of the law.
For bot...

2. Source: CoP_EA_Eng.pdf
Content: domestic duties for the employer as per the Schedule of Accommodation and Domestic
Duties attached to the SEC. It is also stated in Clause 4(b) of the...

3. Source: FDHguideEnglish.pdf
Content: the employment of domestic helpers from outside Hong Kong.
It attempts to answer some of the common questions raised by
the foreign domestic helpers a...

Query: What is the minimum wage?
Retrieved 3 chunks:

1. Source: Concise Guide Minimum Wage.pdf
Content: Effective from 1 May 2026
More information
Statutory Minimum Wage: Reference Guidelines for Employers and Employees
The SMW rate is raised from $42.1 ...

2. Source: Concise Guide Minimum Wage.pdf
Content: Calculation
(i) Minimum wage according to the total number of hours worked for this month:
$10,085.4 (234 hours x $43.1)
(ii) Wages payable in respect...

3. Source: Concise Guide Minimum Wage.pdf
Content: Disabilities under the Statutory Minimum Wage Regime.
The Statutory Minimum Wage (SMW) has implemented an annual review,
with the SMW rate reviewed ea...

Query: What are the rules for employment agencies?
Retrieved 3 chunks:

1. Source: CoP_EA_Eng.pdf
Content: Code of Practice for
Employment Agencies...

2. Source: PGEA_Chapter_3.pdf
Content: CHAPTER 3
CHAPTER 3
Do's and Don'ts for Operating an Employment Agency
Employment Agencies must
3 fully comply with all laws of Hong Kong at all times...

3. Source: PGEA_Chapter_3.pdf
Content: Employment Ordinance, etc.) concerning the employment. If employment agency has doubts
on the provisions, it should advise the job-seekers / employers...
"""

In [ ]:
# rag_pipeline.py
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate
import warnings
warnings.filterwarnings('ignore')

# Loading vector store
print("Loading vectorstore")
embeddings = OllamaEmbeddings(model="tinyllama:latest")
vectorstore = FAISS.load_local(
    "../vectorstore/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

# Initializing TinyLLaMa
print("Initializing TinyLLaMA")
llm = ChatOllama(
    model="tinyllama:latest",
    temperature=0.3,  # Lower to keep it factual
    num_predict=512,
)

# Given prompt
prompt = ChatPromptTemplate.from_template("""
You are a Hong Kong labor law expert. Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't have information about that."

Context: {context}

Question: {question}

Answer:""")

# Creating RAG chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type="stuff",
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

# 5. A test function
def ask(question):
    result = qa_chain.invoke({"query": question})
    print(f"Question: {question}")
    print(f"Answer: {result['result']}")
    print(f"Sources:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"{i}. {doc.metadata.get('source', 'Unknown')}")
    return result

# Small interactive mode
if __name__ == "__main__":
    print("\n" + "="*60)
    print("LABOR POBOT - RAG CHATBOT (TinyLLaMA)")
    print("="*60)
    
    # Test queries
    test_queries = [
        "What are the rights of domestic workers?",
        "What is the minimum wage?",
        "Do domestic workers get rest days?"
    ]
    
    for q in test_queries:
        ask(q)
        print("-" * 60)
    
    # Interactive loop
    print("\nInteractive mode (type 'quit' to exit)")
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ['quit', 'exit']:
            break
        if user_input:
            ask(user_input)